# LocalExplorerAI — Bootstrap Experience Ranker (pilot only)

Notebook này huấn luyện một XGBoost ranker từ CSV hiện có để chạy **pilot có cảnh báo**. Nhãn trong file là `bootstrap_rule_v1`, không phải phản hồi người thật; ETA, slot và sức chứa cũng có fixture/ước tính. Kết quả chỉ đo khả năng học lại quy tắc bootstrap, không chứng minh chất lượng gợi ý thực tế.

Trước khi chạy: push commit chứa notebook và `scripts/train_bootstrap_experience_ranker.py` lên nhánh `main` của repository. Sau đó mở notebook này trong Colab. Khi upload, chọn `experience_ranker_verified.csv` trong `Desktop\New folder` (theo vị trí đã kiểm tra).

Artifact được đánh dấu `production_eligible=false`. API chỉ dùng nó nếu bạn chép bundle vào `models/ai-ranker`, đặt `ENABLE_BOOTSTRAP_RANKER=true` và chấp nhận cảnh báo pilot. Mặc định `rule-baseline-v1` vẫn hoạt động.

In [ ]:
%pip -q install 'xgboost>=3.4.1,<4.0'
# Không nâng cấp NumPy/Pandas riêng để tránh làm lệch runtime Colab.
import xgboost as xgb
print('XGBoost:', xgb.__version__)

In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/LocalExplorerAI')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/NguyenTriBaoThang/LocalExplorerAI.git', str(repo)], check=True)
trainer = repo / 'scripts' / 'train_bootstrap_experience_ranker.py'
assert trainer.is_file(), 'Chưa thấy trainer mới trên GitHub. Hãy commit/push code lên main rồi chạy lại cell clone.'
print('Trainer:', trainer)

## Tải CSV đầu vào

Chọn đúng file bootstrap đã rà soát. Notebook không tự tải dữ liệu khác, không tạo dữ liệu thay thế và không sửa nội dung file.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
assert len(csv_names) == 1, f'Hãy upload đúng một file CSV; hiện nhận được: {csv_names}'
csv_path = Path('/content') / csv_names[0]
assert csv_path.name == 'experience_ranker_verified.csv', f'Notebook này chỉ nhận CSV bootstrap đã rà soát; nhận được {csv_path.name}'
print('Đã nhận:', csv_path.name, f'({csv_path.stat().st_size:,} bytes)')

## Huấn luyện, so sánh và đóng gói

Dùng 15 đặc trưng từ đúng `build_feature_row` của FastAPI; ID, nhãn/phương án giải thích và provenance bị loại khỏi feature. Chia 70/15/15 theo `scenario_family_id` để các biến thể cùng kịch bản không rò sang tập khác. Validation NDCG@5 quyết định runtime được khuyến nghị; test chỉ được tính sau bước đó.

In [ ]:
output_dir = Path('/content/localexplorer_bootstrap_ranker_output')
subprocess.run([
    'python', str(trainer), '--csv', str(csv_path), '--output-dir', str(output_dir)
], cwd=repo, check=True)
import json
metrics = json.loads((output_dir / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics['validation'], ensure_ascii=False, indent=2))
print('Test (chỉ sau lựa chọn validation):')
print(json.dumps(metrics['test_after_validation_selection'], ensure_ascii=False, indent=2))
print('Bundle:', output_dir / 'localexplorer_bootstrap_ranker_bundle.zip')

## Tải model bundle

Bundle có model native XGBoost, checksum, feature schema, provenance, metrics và model card. CSV gốc không được đóng gói vào ZIP. Nếu validation khuyên `rule-baseline-v1`, đó là dấu hiệu XGBoost chưa cải thiện baseline — không diễn giải test yếu-label thành chất lượng sản phẩm.

In [ ]:
files.download(str(output_dir / 'localexplorer_bootstrap_ranker_bundle.zip'))

## Đưa vào pilot API (không phải production)

1. Giải nén trực tiếp các file trong ZIP vào `models/ai-ranker/` (không tạo thư mục con lồng).
2. Cài runtime inference XGBoost theo `apps/api/requirements-ml.txt`; restart API.
3. Chỉ nếu muốn thử nghiệm, đặt `ENABLE_BOOTSTRAP_RANKER=true`; mặc định để `false`.
4. Kiểm tra `GET /api/ml/status`: khi bật, `active_ranker=bootstrap_ml_ranker`, `pilot_runtime_enabled=true`, nhưng `production_eligible=false` và `deployment_allowed=false`.
5. Endpoint planner phải tiếp tục lọc ngân sách, thời gian về, slot/sức chứa, POI khóa và tuyến trước khi đưa ứng viên vào ranker.

Thu feedback hiện có chưa tự tạo bộ nhãn candidate-level đã xác minh: cần lưu các phương án thực sự được hiển thị và ghép chúng với booking/completion/cancellation cùng feedback có đồng ý, sau đó con người rà soát và gán rubric 0–3 trước khi train production.